# Colab GPU 원격 VLM 연산 서버 (Qwen2.5-VL-7B 4-bit)

내 PC의 라벨러 도구(`tools/labeler`)에서 호출할 최고 성능 비전 언어 모델 **Qwen2.5-VL-7B-Instruct**를 Colab T4 GPU에서 4비트 양자화로 단독 기동합니다.

> **[필수 주의사항]**
> 상단 메뉴 **[런타임] → [런타임 유형 변경]**에서 하드웨어 가속기가 반드시 **T4 GPU**로 설정되어 있어야 합니다!

## 1. 패키지 설치 & GPU 환경 검증

In [ ]:
import torch

# 1. GPU 연결 여부 선행 검증
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[중단] GPU(CUDA)가 활성화되지 않았습니다!\n"
        "Colab 상단 메뉴의 [런타임] -> [런타임 유형 변경]에서 'T4 GPU'를 선택하고 다시 실행하세요."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✓ GPU 환경 확인 완료: {gpu_name} (VRAM: {vram_gb:.1f} GB)")

# 2. 의존성 설치 & cloudflared 터널 다운로드
!pip install -q transformers accelerate fastapi uvicorn qwen-vl-utils torchvision bitsandbytes
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("✓ 의존성 패키지 및 터널 도구 설치 완료!")

## 2. OpenAI 호환 Qwen2.5-VL-7B (4-bit) 서버 코드 생성

In [ ]:
%%writefile standalone_vlm_server.py
import base64, io, os, sys
from typing import Any, Dict, List, Optional
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from PIL import Image
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

app = FastAPI(title="Colab Qwen2.5-VL-7B Server")

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
has_cuda = torch.cuda.is_available()
device_str = "cuda" if has_cuda else "cpu"
print(f"[VLM] 7B 모델 가중치 로딩 중 (4-bit={has_cuda}) ...", flush=True)

if has_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map="cpu",
    )

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("[VLM] 7B 모델 로딩 완료! 포트 8090 서비스 준비 완료.", flush=True)

class ChatRequest(BaseModel):
    model: Optional[str] = None
    messages: List[Dict[str, Any]]
    temperature: Optional[float] = 0.2
    max_tokens: Optional[int] = 1024

@app.get("/health")
def health():
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU)"
    return {"status": "ok", "model": MODEL_ID, "gpu": gpu_info}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    try:
        text_prompt = ""
        image_obj = None
        for msg in req.messages:
            content = msg.get("content")
            if isinstance(content, str):
                text_prompt += content + "\n"
            elif isinstance(content, list):
                for part in content:
                    if part.get("type") == "text":
                        text_prompt += part.get("text", "") + "\n"
                    elif part.get("type") == "image_url":
                        url = part.get("image_url", {}).get("url", "")
                        if url.startswith("data:image") and "," in url:
                            b64_data = url.split(",", 1)[1]
                            image_bytes = base64.b64decode(b64_data)
                            image_obj = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        if image_obj is None:
            raise HTTPException(status_code=400, detail="요청에 이미지가 포함되어 있지 않습니다.")

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_obj},
                    {"type": "text", "text": text_prompt.strip() or "이 이미지의 내용을 자세히 설명해주세요."},
                ],
            }
        ]

        text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(conversation)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        if torch.cuda.is_available():
            inputs = inputs.to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens or 1024,
                temperature=req.temperature or 0.2,
                do_sample=(req.temperature or 0.2) > 0,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        return {
            "choices": [
                {
                    "message": {"role": "assistant", "content": output_text},
                    "finish_reason": "stop",
                }
            ]
        }
    except Exception as e:
        print(f"[VLM Error] {e}", flush=True)
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8090, log_level="info")


## 3. VLM 서버 기동 & Cloudflare 터널 URL 생성 (실행 후 유지)

In [ ]:
import subprocess, time, re, urllib.request, sys

# 1. VLM 서버 백그라운드 기동 (로그를 server.log에 실시간 저장)
print("[1/3] Qwen2.5-VL-7B 모델 다운로드 및 GPU 로딩 시작... (최초 다운로드 시 약 1~2분 소요)")
with open("server.log", "w") as f:
    server_proc = subprocess.Popen(["python", "standalone_vlm_server.py"], stdout=f, stderr=subprocess.STDOUT)

# 2. 서버가 정상 기동(health 200)될 때까지 대기
ready = False
for sec in range(180):  # 최대 3분 대기
    if server_proc.poll() is not None:
        print("\n[오류 발생] 서버 프로세스가 비정상 종료되었습니다! 로그 내용:")
        with open("server.log", "r") as f:
            print(f.read())
        raise SystemExit(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8090/health", timeout=1) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    if sec % 10 == 0 and sec > 0:
        print(f"... 모델 로딩 중 ({sec}초 경과) ...")
    time.sleep(1)

if not ready:
    print("\n[시간 초과] 모델 로딩 대기 시간(3분)이 초과되었습니다. server.log:")
    with open("server.log", "r") as f:
        print(f.read())
    raise SystemExit(1)

print("✓ Qwen2.5-VL-7B 모델 로딩 완료 및 8090 포트 활성화 확인!")

# 3. Cloudflare 터널 기동
print("[2/3] Cloudflare 공용 보안 터널 생성 중...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8090"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print("\n===================================================================")
    print(f"[3/3] Colab Qwen2.5-VL-7B 서버가 완벽하게 준비되었습니다!")
    print(f"터널 기본 주소: {tunnel_url}")
    print(f"\n[내 PC 적용 방법]")
    print(f"내 PC의 'tools/labeler/.env' 파일에 아래 두 줄을 복사해 넣으세요:")
    print(f"-------------------------------------------------------------------")
    print(f"VLM_PROVIDER=local")
    print(f"VLM_BASE_URL={tunnel_url}/v1")
    print(f"-------------------------------------------------------------------")
    print("이제 내 PC 라벨러 웹 또는 배치 작업을 돌리면 502 에러 없이 즉시 정상 처리됩니다.")
    print("===================================================================\n")
    # 터널 유지
    tunnel_proc.wait()
else:
    print("터널 생성 실패. 로그를 확인하세요.")

[1/3] Qwen2.5-VL-7B 모델 다운로드 및 GPU 로딩 시작... (최초 다운로드 시 약 1~2분 소요)
... 모델 로딩 중 (10초 경과) ...
... 모델 로딩 중 (20초 경과) ...
... 모델 로딩 중 (30초 경과) ...
... 모델 로딩 중 (40초 경과) ...
... 모델 로딩 중 (50초 경과) ...
... 모델 로딩 중 (60초 경과) ...
... 모델 로딩 중 (70초 경과) ...
✓ Qwen2.5-VL-7B 모델 로딩 완료 및 8090 포트 활성화 확인!
[2/3] Cloudflare 공용 보안 터널 생성 중...

[3/3] Colab Qwen2.5-VL-7B 서버가 완벽하게 준비되었습니다!
터널 기본 주소: https://legislative-cube-logo-britney.trycloudflare.com

[내 PC 적용 방법]
내 PC의 'tools/labeler/.env' 파일에 아래 두 줄을 복사해 넣으세요:
-------------------------------------------------------------------
VLM_PROVIDER=local
VLM_BASE_URL=https://legislative-cube-logo-britney.trycloudflare.com/v1
-------------------------------------------------------------------
이제 내 PC 라벨러 웹 또는 배치 작업을 돌리면 502 에러 없이 즉시 정상 처리됩니다.

